## Article Cleaning Pipeline Overview

- Web-crawled news usually contains HTML noise, boilerplate text, duplicates, and off-topic content.
- Without cleaning, entity extraction, sentiment analysis, and topic modeling become unstable.

### Main output
- Final dataframe: `clean_articles_df`
- Consistent schema, cleaner text, fewer duplicates, and stronger topic relevance

### Downstream use
The cleaned data can be used for industry/company extraction, impact direction analysis, mechanism tagging (automation/augmentation), and topic discovery.


In [ ]:
import re
import html
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
df_news_final_project = pd.read_parquet('https://storage.googleapis.com/msca-bdp-data-open/news_final_project/news_final_project.parquet', engine='pyarrow')
df_news_final_project.shape

In [ ]:
df_news_final_project.head()

## Step 1: Basic Schema Handling

### Objective
Validate required columns and normalize core text fields.

### Operations
- Check required columns (`title` + `content`/`text`).
- Safely convert text fields to strings and trim whitespace.
- Remove rows where article body is missing or empty.

### Why it matters
This is the foundation for all later steps and prevents type/empty-text issues in regex cleaning and filtering.

### Output
A valid `articles_df` with non-empty article text and row counts after missing-value removal.


In [ ]:
# Step 1: Basic schema handling

articles_df = df_news_final_project.copy()
articles_df['title'] = articles_df['title'].fillna('').astype(str).str.strip()
articles_df['text'] = articles_df['text'].fillna('').astype(str).str.strip()

pipeline_stats = {}
pipeline_stats['raw_rows'] = len(articles_df)

# Drop rows where content is missing/empty after stripping whitespace
articles_df = articles_df[articles_df['text'].str.len() > 0].copy()
pipeline_stats['after_missing_rows'] = len(articles_df)

print(f"Raw rows: {pipeline_stats['raw_rows']}")
print(f"After missing-value drop: {pipeline_stats['after_missing_rows']}")
articles_df[['title', 'text']].head(2)


## Step 2: Remove HTML and Crawl Artifacts

### Objective
Remove web-crawl formatting noise while preserving semantic article content.

### Operations in `clean_text()`
- Decode HTML entities (for example `&amp;`, `&nbsp;`).
- Strip HTML tags (`<...>`).
- Remove inline URLs.
- Remove common boilerplate phrases (`subscribe`, `advertisement`, `read more`, etc.).
- Remove obvious JS/CSS fragments.
- Normalize repeated whitespace and extra line breaks.

### Why it matters
This step improves text quality and reduces non-semantic tokens that can distort keyword matching and NLP models.

### Output
Two cleaned columns: `clean_title` and `clean_content`, with empty cleaned rows removed.


In [ ]:
# Step 2: Remove HTML / web crawl artifacts
boilerplate_patterns = [
    r'\bsubscribe\s+now\b',
    r'\badvertisement\b',
    r'\bclick\s+here\b',
    r'\bread\s+more\b',
    r'\bsign\s+up\b',
    r'\bnewsletter\b',
    r'\bcookie\s+policy\b',
    r'\ball\s+rights\s+reserved\b',
    r'©',
]

boilerplate_regex = re.compile('|'.join(boilerplate_patterns), flags=re.IGNORECASE)
html_tag_regex = re.compile(r'<[^>]+>')
url_regex = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)
js_css_regex = re.compile(
    r'(function\s*\(|var\s+\w+\s*=|let\s+\w+\s*=|const\s+\w+\s*=|\{\s*\}|;\s*$|\.css\b|\.js\b)',
    flags=re.IGNORECASE
)

def clean_text(text: str) -> str:
    if pd.isna(text):
        return ''

    text = str(text)
    text = html.unescape(text)
    text = html_tag_regex.sub(' ', text)
    text = url_regex.sub(' ', text)
    text = js_css_regex.sub(' ', text)
    text = boilerplate_regex.sub(' ', text)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text).strip()
    return text

articles_df['clean_title'] = articles_df['title'].map(clean_text)
articles_df['clean_content'] = articles_df['text'].map(clean_text)
articles_df = articles_df[articles_df['clean_content'].str.len() > 0].copy()

print(f"After Step 2 cleaning: {len(articles_df)}")
articles_df[['clean_title', 'clean_content']].head(2)


## Step 3: Length Filtering

### Objective
Remove very short, low-information articles.

### Operations
- Compute `content_char_len` and `word_count` as quality indicators.
- Apply a minimum character threshold (for example `MIN_CONTENT_CHARS = 200`).

### Why it matters
Very short texts are often snippets, navigation fragments, or weak evidence for industry-impact analysis.

### Output
A higher-information subset plus length statistics for sanity checks.


In [ ]:
# Step 3: Length filtering
MIN_CONTENT_CHARS = 200  # easy to tune

articles_df['content_char_len'] = articles_df['clean_content'].str.len()
articles_df['word_count'] = articles_df['clean_content'].str.split().map(len)

articles_df = articles_df[articles_df['content_char_len'] >= MIN_CONTENT_CHARS].copy()
pipeline_stats['after_short_filter_rows'] = len(articles_df)

print(f"After short-article filter (chars >= {MIN_CONTENT_CHARS}): {pipeline_stats['after_short_filter_rows']}")
articles_df[['content_char_len', 'word_count']].describe().T


## Step 4: Exact Deduplication

### Objective
Remove exact duplicate articles to avoid double-counting the same content.

### Operations
- Primary pass: deduplicate by `clean_content` (keep first).
- Optional stricter pass: deduplicate by `clean_title + clean_content`.

### Why it matters
News datasets commonly contain mirrored/reposted copies. Keeping them would bias company/industry frequency analysis.

### Output
A deduplicated dataset with row counts after exact duplicate removal.


In [ ]:
# Step 4: Exact duplicate removal
DROP_DUP_BY_TITLE_AND_CONTENT = True  # optional exact dedup pass

articles_df = articles_df.drop_duplicates(subset=['clean_content'], keep='first').copy()

if DROP_DUP_BY_TITLE_AND_CONTENT:
    articles_df = articles_df.drop_duplicates(subset=['clean_title', 'clean_content'], keep='first').copy()

pipeline_stats['after_exact_dedup_rows'] = len(articles_df)
print(f"After exact dedup: {pipeline_stats['after_exact_dedup_rows']}")


## Step 5: Relevance Filtering

### Objective
Keep only AI/ML/Data Science-relevant articles and remove clearly off-topic content.

### Rule design
- `include_keywords`: AI, machine learning, deep learning, LLM, generative AI, NLP, computer vision, etc.
- `exclude_keywords`: sports, politics, weather, celebrity gossip, etc.
- Case-insensitive matching over `title + content`.

### Filtering logic
- Keep a row only if it matches at least one include keyword.
- Rows that only match exclude topics and do not show AI relevance are removed.

### Why it matters
This step aligns the corpus with the project objective: identifying AI impact on industries and companies.

### Output
A topic-focused corpus with row counts after relevance filtering.


In [ ]:
# Step 5: Relevance filtering with include/exclude keywords
include_keywords = [
    'artificial intelligence',
    'machine learning',
    'deep learning',
    'data science',
    'llm', 'large language model', 'large language models',
    'generative ai',
    'automation',
    'neural network', 'neural networks',
    'nlp',
    'computer vision'
]

exclude_keywords = [
    'sports', 'sport',
    'politics', 'political',
    'weather',
    'celebrity', 'gossip'
]

include_regex = re.compile('|'.join(re.escape(k) for k in include_keywords), flags=re.IGNORECASE)
exclude_regex = re.compile('|'.join(re.escape(k) for k in exclude_keywords), flags=re.IGNORECASE)

articles_df['search_text'] = (articles_df['clean_title'].fillna('') + ' ' + articles_df['clean_content'].fillna('')).str.lower()
articles_df['include_hits'] = articles_df['search_text'].str.count(include_regex)
articles_df['exclude_hits'] = articles_df['search_text'].str.count(exclude_regex)

# Keep only rows with at least one AI-related include keyword.
# If an article only matches excluded topics and has no include signal, it is removed.
articles_df = articles_df[articles_df['include_hits'] > 0].copy()

pipeline_stats['after_relevance_rows'] = len(articles_df)
print(f"After relevance filtering: {pipeline_stats['after_relevance_rows']}")
articles_df[['include_hits', 'exclude_hits']].describe().T


## Final Output

### Final dataframe
The final output is `clean_articles_df` (or `final`) with the core schema:
- `article_id`
- `date`
- `title`
- `text`

### Reproducibility
The notebook prints row counts at each stage (raw -> missing-value drop -> short-text filter -> dedup -> relevance -> final) for transparent reporting.

### Next step
Use this output for entity extraction (industry/company), impact direction classification, mechanism tagging, and topic modeling.


In [ ]:
clean_articles_df = articles_df[['article_id', 'date', 'clean_title', 'clean_content']].copy()
clean_articles_df = clean_articles_df.rename(columns={
    'clean_title': 'title',
    'clean_content': 'content'
}).reset_index(drop=True)

print(f"Final row count: {len(clean_articles_df)}")
clean_articles_df.head()


In [ ]:
clean_articles_df.to_parquet("clean_articles_step1.parquet", index=False)